In [1]:
from pathlib import Path
import re, json
from collections import Counter

TRACE_DIR = Path("traces_clean")
OUT_JSONL = Path("sfence_vma_path_summaries.jsonl")
OUT_TXT = Path("sfence_vma_path_summaries.txt")

def split_trace_blocks(text):
    lines = text.splitlines()
    starts = [i for i, line in enumerate(lines) if line.strip().startswith("(trace")]
    if not starts:
        return [text] if text.strip() else []
    blocks = []
    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else len(lines)
        block = "\n".join(lines[start:end]).strip()
        if block:
            blocks.append(block)
    return blocks

def is_tool_error(text):
    low = text.lower()
    return any(x in low for x in ["panic", "panicked", "dumped core", "timeout", "error:"])

def is_illegal_path(text):
    return "write-reg |mcause| nil (_ struct (|bits| #x0000000000000002))" in text

def is_success_path(text):
    return "RETIRE_SUCCESS" in text and not is_illegal_path(text) and not is_tool_error(text)

def priv_from_name(name):
    m = re.search(r"__(Machine|Supervisor|User|VirtualUser|VirtualSupervisor)__", name)
    return m.group(1) if m else "Unknown"

def experiment_from_name(name):
    if name.startswith("sfence__"):
        return "sfence_vma"
    if name.startswith("sinval__"):
        return "sinval_vma_execute_as_sfence"
    if name.startswith("virtual_probe__"):
        return "virtual_privilege_probe"
    return "unknown"

def instruction_from_name(name):
    if name.startswith("sfence__") or name.startswith("virtual_probe__"):
        base = "sfence.vma"
    elif name.startswith("sinval__"):
        base = "sinval.vma"
    else:
        base = "unknown"

    rs1 = "x5" if "rs1_x5" in name else "x0"
    rs2 = "x6" if "rs2_x6" in name else "x0"
    return f"{base} {rs1}, {rs2}"

def operand_info(name):
    rs1 = "x5" if "rs1_x5" in name else "x0"
    rs2 = "x6" if "rs2_x6" in name else "x0"
    addr_filter = "Some(address from x5)" if rs1 != "x0" else "None/all addresses"
    asid_filter = "Some(ASID from x6)" if rs2 != "x0" else "None/all ASIDs"
    return rs1, rs2, addr_filter, asid_filter

def short(line, n=220):
    line = line.strip()
    if len(line) <= n:
        return line
    return line[:n] + " ... [truncated]"

def important_events(block):
    keys = [
        "assume-reg |cur_privilege|",
        "assume-reg |x5|",
        "assume-reg |x6|",
        "read-reg |cur_privilege|",
        "read-reg |mstatus|",
        "read-reg |x5|",
        "read-reg |x6|",
        "read-reg |mcause|",
        "write-reg |mcause|",
        "write-reg |mstatus|",
        "write-reg |cur_privilege|",
        "write-reg |nextPC|",
        "read-reg |tlb|",
        "write-reg |tlb|",
        "Retired",
        "Error:",
        "error:",
        "panic",
        "panicked",
    ]
    out = []
    for line in block.splitlines():
        s = line.strip()
        if any(k in s for k in keys):
            out.append(short(s))
    return out[:120]

def reason(priv, status, exp):
    if status == "tool_error":
        if exp == "sinval_vma_execute_as_sfence":
            return "SINVAL.VMA is present in the Sail source, but this artifact/assembler requires the Svinval extension for the assembly instruction, so no clean trace is produced."
        if exp == "virtual_privilege_probe":
            return "Virtual privilege values are not accepted by the current Sailor/Isla IR, so this is a model/tool limitation."
        return "The run hit an Isla/Sailor tool error before producing a clean architectural path."

    if status == "trapped":
        if priv == "User":
            return "User mode SFENCE.VMA is illegal."
        if priv == "Supervisor":
            return "Supervisor mode traps on the path where mstatus.TVM is 1."
        return "The instruction traps with illegal instruction."

    if status == "success":
        if priv == "Machine":
            return "Machine mode executes SFENCE.VMA and retires successfully."
        if priv == "Supervisor":
            return "Supervisor mode executes SFENCE.VMA and retires successfully on the path where mstatus.TVM is 0."
        return "The instruction retires successfully."

    return "The trace was generated, but the outcome was not parsed clearly."

rows = []

for path in sorted(TRACE_DIR.glob("*.trace")):
    text = path.read_text(errors="replace")
    blocks = split_trace_blocks(text)

    exp = experiment_from_name(path.name)
    priv = priv_from_name(path.name)
    instr = instruction_from_name(path.name)
    rs1, rs2, addr_filter, asid_filter = operand_info(path.name)

    for i, block in enumerate(blocks):
        terr = is_tool_error(block)
        trapped = is_illegal_path(block)
        success = is_success_path(block)

        if terr:
            status = "tool_error"
            exception = "tool_error"
        elif trapped:
            status = "trapped"
            exception = "illegal_instruction"
        elif success:
            status = "success"
            exception = "none"
        else:
            status = "unknown"
            exception = "unknown"

        if exp == "sfence_vma":
            if status == "success":
                path_name = f"{priv}-mode SFENCE.VMA succeeds with {rs1}/{rs2}"
            elif status == "trapped":
                path_name = f"{priv}-mode SFENCE.VMA traps with {rs1}/{rs2}"
            else:
                path_name = f"{priv}-mode SFENCE.VMA {status} with {rs1}/{rs2}"
        elif exp == "sinval_vma_execute_as_sfence":
            path_name = f"{priv}-mode SINVAL.VMA {status} with {rs1}/{rs2}"
        elif exp == "virtual_privilege_probe":
            path_name = f"{priv} SFENCE.VMA virtual privilege probe"
        else:
            path_name = "unknown SFENCE.VMA path"

        summary = (
            f"`{instr}` uses address filter {addr_filter} and ASID filter {asid_filter}. "
            f"{reason(priv, status, exp)}"
        )

        rows.append({
            "experiment": exp,
            "path_id": f"{path.stem}__path_{i:03d}",
            "trace_file": str(path),
            "path_name": path_name,
            "privilege": priv,
            "instruction": instr,
            "rs1": rs1,
            "rs2": rs2,
            "address_filter": addr_filter,
            "asid_filter": asid_filter,
            "status": status,
            "exception": exception,
            "tool_error": terr,
            "summary": summary,
            "important_trace_events": important_events(block),
        })

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

lines = []
for idx, row in enumerate(rows, 1):
    lines.append(f"Path {idx}: {row['path_name']}")
    lines.append(f"Path ID: {row['path_id']}")
    lines.append(f"Experiment: {row['experiment']}")
    lines.append(f"Trace file: {row['trace_file']}")
    lines.append("")
    lines.append("Condition:")
    lines.append(f"- Privilege: {row['privilege']}.")
    lines.append(f"- Instruction: {row['instruction']}.")
    lines.append(f"- rs1: {row['rs1']}.")
    lines.append(f"- rs2: {row['rs2']}.")
    lines.append(f"- Address filter: {row['address_filter']}.")
    lines.append(f"- ASID filter: {row['asid_filter']}.")
    lines.append("")
    lines.append("Outcome:")
    lines.append(f"- Status: {row['status']}.")
    lines.append(f"- Exception: {row['exception']}.")
    lines.append(f"- Tool error: {row['tool_error']}.")
    lines.append("")
    lines.append("Summary:")
    lines.append(row["summary"])
    lines.append("")
    lines.append("Important trace events:")
    for e in row["important_trace_events"]:
        lines.append(f"- {e}")
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

OUT_TXT.write_text("\n".join(lines), encoding="utf-8")

print("total rows:", len(rows))
print("by experiment:", Counter(r["experiment"] for r in rows))
print("by status:", Counter(r["status"] for r in rows))
print("wrote:", OUT_JSONL)
print("wrote:", OUT_TXT)

total rows: 24
by experiment: Counter({'sfence_vma': 16, 'sinval_vma_execute_as_sfence': 6, 'virtual_privilege_probe': 2})
by status: Counter({'success': 8, 'trapped': 8, 'tool_error': 8})
wrote: sfence_vma_path_summaries.jsonl
wrote: sfence_vma_path_summaries.txt


In [2]:
from pathlib import Path
import re

INPUT_FILE = Path("rule.txt")
OUTPUT_FILE = Path("rule_clean.txt")

text = INPUT_FILE.read_text(errors="ignore")

rule_pattern = re.compile(
    r"(?m)^RULE(?:\s+Path)?\s+\d+\s*:\s*"
)

matches = list(rule_pattern.finditer(text))

if not matches:
    raise ValueError("No RULE lines found in rule.txt")

blocks = []

for i, m in enumerate(matches):
    start = m.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    block = text[start:end].strip()
    blocks.append(block)

seen = set()
clean_rules = []

for block in blocks:
    body = re.sub(r"^RULE(?:\s+Path)?\s+\d+\s*:\s*", "", block, flags=re.IGNORECASE).strip()

    key = re.sub(r"\s+", " ", body).strip().lower()

    if key in seen:
        continue

    seen.add(key)
    clean_rules.append(body)

output_lines = []

for i, body in enumerate(clean_rules, 1):
    output_lines.append(f"RULE {i}: {body}")

OUTPUT_FILE.write_text("\n".join(output_lines) + "\n")

print("input rules:", len(blocks))
print("unique rules:", len(clean_rules))
print("removed duplicates:", len(blocks) - len(clean_rules))
print("wrote:", OUTPUT_FILE)

input rules: 24
unique rules: 24
removed duplicates: 0
wrote: rule_clean.txt
